In [1]:
import os, json, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
os.makedirs("../models", exist_ok=True)

# CV + scoring (primary = F1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORERS = {"f1": "f1", "accuracy": "accuracy", "precision": "precision", "recall": "recall", "roc_auc": "roc_auc"}
PRIMARY = "f1"


In [2]:
# Either load your canonical engineered dataset (features + Revenue)...
df = pd.read_csv("../data/train_data_with_engineered.csv")
y = df["Revenue"].astype(int)
X_full = df.drop(columns=["Revenue"])

# ...and the selected feature list from your feature selection step
with open("../results/feature_set_plan.json", "r") as f:
    plan = json.load(f)

best_name  = plan["best_overall"]["name"]       # e.g., "num_pruned_engineered_plus_temporal"
best_cols  = plan["best_overall"]["columns"]    # exact list of columns
X = X_full[[c for c in best_cols if c in X_full.columns]].copy()

print(f"Using feature set: {best_name} | {X.shape}")


Using feature set: num_pruned_engineered_plus_temporal | (9864, 11)


In [3]:
# Heuristic detection; override CAT_COLS manually if needed
CAT_COLS = [c for c in X.columns if X[c].dtype == "object" or str(X[c].dtype).startswith(("category","bool"))]
NUM_COLS = [c for c in X.columns if c not in CAT_COLS]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
], remainder="drop")

print(f"NUM: {len(NUM_COLS)} | CAT: {len(CAT_COLS)}")


NUM: 10 | CAT: 1


In [6]:
def run_search(name, estimator, param_grid, search="grid", n_iter=20, refit=PRIMARY):
    """
    Runs a CV search with a preprocessing pipeline.
    - search="grid"  -> GridSearchCV(param_grid=...)
    - search="random"-> RandomizedSearchCV(param_distributions=..., n_iter=n_iter)
    Returns: (best_estimator_pipeline, result_row_dict)
    """
    pipe = Pipeline([("prep", preprocessor), ("clf", estimator)])
    grid = {f"clf__{k}": v for k, v in param_grid.items()}

    if search == "grid":
        searcher = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            cv=cv,
            scoring=PRIMARY,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )
    else:  # "random"
        searcher = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=grid,
            n_iter=n_iter,
            cv=cv,
            scoring=PRIMARY,
            n_jobs=-1,
            verbose=1,
            random_state=42,
            refit=True,
        )

    searcher.fit(X, y)

    # Evaluate the best pipeline with multiple metrics (no leakage: prep is inside)
    cv_res = cross_validate(
        searcher.best_estimator_, X, y, cv=cv, scoring=SCORERS, n_jobs=-1, return_train_score=False
    )
    row = {
        "Model": name,
        "BestParams": searcher.best_params_,
        **{f"mean_{k}": np.mean(v) for k, v in cv_res.items() if k.startswith("test_")},
        **{f"std_{k}": np.std(v) for k, v in cv_res.items() if k.startswith("test_")},
    }
    return searcher.best_estimator_, row


In [7]:
results = []
best_models = {}

# 4a) Logistic Regression — L1
best_models["LogReg_L1"], row = run_search(
    "LogReg_L1",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    {"penalty": ["l1"], "solver": ["liblinear", "saga"], "C": [0.001,0.01,0.1,1,10,100]},
    search="grid"
)
results.append(row)

param_grid_logreg = {
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "C": [0.001, 0.01, 0.1, 1, 10, 100]
}   
# 4b) Logistic Regression — L2
best_models["LogReg_L2"], row = run_search(
    "LogReg_L2",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    param_grid_logreg,
    search="grid"
)
results.append(row)


# 4c) Logistic Regression — Elastic Net
param_grid_logreg_en = {
    "penalty": ["elasticnet"],
    "solver": ["saga"],
    "C": [0.001, 0.01, 0.1, 1, 10],
    "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
}
best_models["LogReg_EN"], row = run_search(
    "LogReg_EN",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    param_grid_logreg_en,
    search="grid"
)
results.append(row)

# 4d) Logistic Regression — None
param_grid_logreg_none = {
    "penalty": [None],
    "solver": ["lbfgs"]
}
best_models["LogReg_None"], row = run_search(
    "LogReg_None",
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
    param_grid_logreg_none,
    search="grid"
)
results.append(row)

# 4e) Random Forest
param_grid_rf = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}       
best_models["RF"], row = run_search(
    "RandomForest",
    RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced"),
    param_grid_rf,
    search="random", n_iter=24
)
results.append(row)

from xgboost import XGBClassifier
pos = y.sum(); neg = len(y) - pos
spw = neg / max(pos, 1)

param_grid_xgb = {
    "n_estimators": [300, 600],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.02, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0],
    "reg_alpha": [0.0, 0.5, 1.0],
}
best_models["XGB"], row = run_search(
    "XGBoost",
    XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=spw,
    ),
    param_grid_xgb,
    search="random", n_iter=32
)
results.append(row)


Fitting 5 folds for each of 12 candidates, totalling 60 fits


Fitting 5 folds for each of 24 candidates, totalling 120 fits
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 24 candidates, totalling 120 fits
Fitting 5 folds for each of 32 candidates, totalling 160 fits


In [9]:
leaderboard = pd.DataFrame(results).sort_values("mean_test_f1", ascending=False)
display(leaderboard[["Model","BestParams","mean_test_f1","std_test_f1","mean_test_accuracy","mean_test_precision","mean_test_recall","mean_test_roc_auc"]])

# Save top-1 pipeline
from joblib import dump
top_name = leaderboard.iloc[0]["Model"]
print(f"saved: ../models/best_pipeline_{top_name}.joblib")


,Model,BestParams,mean_test_f1,std_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,mean_test_roc_auc
4,RandomForest,"{'clf__n_estimators': 300, 'clf__min_samples_s...",0.677632,0.008429,0.889497,0.617710,0.750961,0.928265
5,XGBoost,"{'clf__subsample': 0.7, 'clf__reg_lambda': 1.0...",0.659836,0.016214,0.873885,0.566567,0.790290,0.928055
0,LogReg_L1,"{'clf__C': 0.001, 'clf__penalty': 'l1', 'clf__...",0.621249,0.019319,0.886558,0.654780,0.604089,0.883765
1,LogReg_L2,"{'clf__C': 0.001, 'clf__penalty': 'l1', 'clf__...",0.621249,0.019319,0.886558,0.654780,0.604089,0.883765
3,LogReg_None,"{'clf__penalty': None, 'clf__solver': 'lbfgs'}",0.610703,0.015315,0.852494,0.516236,0.747695,0.903109
2,LogReg_EN,"{'clf__C': 1, 'clf__l1_ratio': 0.9, 'clf__pena...",0.610195,0.015506,0.852798,0.517122,0.744417,0.903204


saved: ../models/best_pipeline_RandomForest.joblib
